# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [5]:
#%pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-groq langchain-huggingface langchain-elasticsearch jq lark elasticsearch

In [6]:
import os
from dotenv import load_dotenv, find_dotenv

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

from langchain_community.vectorstores.elasticsearch import ElasticsearchStore

In [7]:
_ = load_dotenv(find_dotenv())

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')
ELASTIC_API_KEY = os.getenv('ELASTIC_API_KEY')
ELASTIC_CLOUD_ID = os.getenv('ELASTIC_CLOUD_ID')

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [8]:
# Create HuggingFace Hub embedding model (remote inference, no GPU needed locally)
embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-large-en-v1.5",
    huggingfacehub_api_token=HF_TOKEN,
)

# Set a meaningful index name
INDEX_NAME = "workplace-docs-bge"

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_url=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [9]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 200 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [10]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
from datetime import datetime


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None

    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

# Use character-based splitting (no tiktoken / OpenAI tokenizer dependency)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
)

docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into chunks of 800 characters, we will index data into Elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use the **Elasticsearch endpoint URL** and **API Key** from our Elastic Cloud Serverless project, along with a **HuggingFace Hub** embedding model for vectorization and **Groq's llama-3.3-70b** as the language model.

In [11]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Clean docs metadata
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()

    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()

    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

# Create HuggingFace Hub embeddings (1024-dim, stronger semantic understanding)
embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-large-en-v1.5",
    huggingfacehub_api_token=HF_TOKEN,
)

# Create vectorstore from cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_url=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create Groq LLM (llama-3.3-70b)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY,
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [12]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

# Per-document formatter (used inside safe_combine_docs)
LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Final RAG prompt (takes {context} + {question})
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Use the following context to answer the question. Be concise and accurate.

Context:
{context}

Question: {question}

Answer:
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

def multi_query_rag(question):
    """Generate + retrieve + answer"""
    queries = MULTI_QUERY_PROMPT | llm | StrOutputParser()
    all_docs = []
    for q in queries.invoke(question).split("\n"):
        q = q.strip()
        if q:
            docs = self_healing_retriever(q)
            all_docs.extend(docs[:3])

    return rag_chain.invoke({"question": question, "context": safe_combine_docs(all_docs)})

print("---- Answer ----")
print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ["What is the function of NASA's marketing team in advancing space-related merchandise and experiences?", ' ', 'How does the sales division at NASA contribute to the development and promotion of space travel and exploration offerings?', ' ', "In what ways does NASA's commercial outreach sector support the growth of space exploration initiatives through product and service promotion?"]
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-a69c61.es.europe-west1.gcp.elastic.cloud:443/workplace-docs-bge/_search?_source_includes=metadata,text [status:200 duration:0.031s]
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-a69c61.es.europe-west1.gcp.elastic.cloud:443/workplace-docs-bge/_search?_source_i

The NASA sales team refers to the North America South America region sales team, which includes the United States, Canada, Mexico, Central, and South America. It is led by two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [13]:
print("---- Iteration 2: Work From Home Policy ----")
print(multi_query_rag("What is the company's work from home policy?"))

---- Iteration 2: Work From Home Policy ----


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What flexible work options are available to employees at the company?', 'What types of telecommuting or work-from-home policies does the company have in place?', 'What benefits or programs does the company offer to support employees who work remotely or have non-traditional schedules?']
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-a69c61.es.europe-west1.gcp.elastic.cloud:443/workplace-docs-bge/_search?_source_includes=metadata,text [status:200 duration:0.024s]
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-a69c61.es.europe-west1.gcp.elastic.cloud:443/workplace-docs-bge/_search?_source_includes=metadata,text [status:200 duration:0.022s]
INFO:elastic_transport.transport:POST https://m

The company's work-from-home policy provides guidelines and support for employees to conduct their work remotely. The policy includes provisions for equipment and resources, workspace, communication, health and well-being, and performance expectations. Employees are expected to maintain the same level of performance and productivity as if they were working in the office, and are responsible for maintaining and protecting the company's equipment and data. The policy also encourages employees to prioritize their health and well-being while working from home, and to maintain regular communication with their supervisors, colleagues, and team members. 

Key aspects of the policy include:
- The company provides necessary equipment and resources for remote work.
- Employees are responsible for creating a comfortable and safe workspace.
- Employees must maintain regular communication with supervisors, colleagues, and team members.
- Employees are expected to maintain their regular work hours a

In [14]:
print("---- Iteration 3: Expense Policy ----")
print(multi_query_rag("What are the rules for submitting and approving expense reports?"))

---- Iteration 3: Expense Policy ----


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What are the procedures for submitting and approving expense reports in the company?', ' ', 'What are the rules and regulations governing expense report submission, review, and approval processes?', ' ', 'How do I properly submit an expense report and what are the steps involved in getting it approved by the management or accounting department?']
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-a69c61.es.europe-west1.gcp.elastic.cloud:443/workplace-docs-bge/_search?_source_includes=metadata,text [status:200 duration:0.024s]
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-a69c61.es.europe-west1.gcp.elastic.cloud:443/workplace-docs-bge/_search?_source_includes=metadata,text [status:200 dur

There are no rules for submitting and approving expense reports mentioned in the provided context. The context covers various topics such as onboarding, benefits, tax elections, and company policies, but it does not address expense reports.


## Improving the Output

The raw `multi_query_rag` output has two problems:
- **Verbose INFO logs** from LangChain and Elasticsearch clutter every response
- **Plain `print()` output** with no visual structure

The `pretty_ask` function below addresses both:
1. **Silences INFO logs** during execution using `logging.disable`, then restores them
2. **Renders the response as styled HTML** via `IPython.display` — question in a highlighted header, answer in a readable card
3. **Shows retrieved source documents** in a collapsible section so you can verify what the LLM was given

In [16]:
import logging
from IPython.display import display, HTML

def pretty_ask(question):
    # Silence INFO logs during execution
    logging.disable(logging.INFO)
    try:
        queries = MULTI_QUERY_PROMPT | llm | StrOutputParser()
        all_docs = []
        for q in queries.invoke(question).split("\n"):
            q = q.strip()
            if q:
                docs = self_healing_retriever(q)
                all_docs.extend(docs[:3])
        answer = rag_chain.invoke({"question": question, "context": safe_combine_docs(all_docs)})
    finally:
        logging.disable(logging.NOTSET)

    # Deduplicate passages by content
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)

    # Build one collapsible block per unique passage
    passages_html = ""
    for i, doc in enumerate(unique_docs):
        source = doc.metadata.get("name") or doc.metadata.get("source", f"Passage {i+1}")
        passage = doc.page_content.replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        passages_html += f"""
        <details style="margin-bottom:6px; border:1px solid #dee2e6; border-radius:4px;">
            <summary style="cursor:pointer; padding:6px 10px; background:#e9ecef;
                            font-size:12px; font-weight:600; color:#495057;">
                📄 {source}
            </summary>
            <div style="padding:8px 12px; font-size:12px; color:#495057;
                        line-height:1.5; background:#fff;">
                {passage}
            </div>
        </details>
        """

    display(HTML(f"""
    <div style="font-family: sans-serif; max-width: 800px; margin: 12px 0;">
        <div style="background:#1f3864; color:white; padding:12px 16px; border-radius:8px 8px 0 0;">
            <span style="font-size:13px; opacity:0.7;">Question</span><br>
            <strong style="font-size:16px;">{question}</strong>
        </div>
        <div style="background:#f8f9fa; border:1px solid #dee2e6;
                    padding:16px; border-radius:0 0 8px 8px;">
            <p style="margin:0 0 16px 0; font-size:15px; line-height:1.6; color:#212529;">
                {answer.replace(chr(10), "<br>")}
            </p>
            <details>
                <summary style="cursor:pointer; font-size:12px; color:#6c757d; margin-bottom:8px;">
                    Retrieved passages ({len(unique_docs)})
                </summary>
                <div style="margin-top:8px;">
                    {passages_html}
                </div>
            </details>
        </div>
    </div>
    """))

# --- Demo ---
pretty_ask("What is the NASA sales team?")
pretty_ask("What is the company's work from home policy?")
pretty_ask("What are the rules for submitting and approving expense reports?")

## Summary

### What We Built
A retrieval-augmented generation (RAG) chatbot that answers questions over internal company documents, using:
- **Groq** (`llama-3.3-70b-versatile`) as the LLM — free, fast, no local GPU required
- **BAAI/bge-large-en-v1.5** via HuggingFace Hub for remote embeddings (1024-dim)
- **Elasticsearch Serverless** as the vector store
- **MultiQuery Retriever** to generate query variants and broaden document recall
- **Prettified HTML output** with collapsible source passages per answer

### What We Learned
- MultiQuery Retriever improves recall by rephrasing the same question multiple ways before searching, reducing the chance of missing relevant passages due to wording mismatch
- Embedding model choice directly impacts retrieval quality — `all-MiniLM-L6-v2` misidentified "NASA" (a sales region acronym) as the space agency; switching to `bge-large-en-v1.5` resolved it
- Elasticsearch stores vectors and serves results via HTTP — it consumes zero LLM tokens; only the LangChain calls to Groq do
- Changing embedding models requires re-indexing with a new index name to avoid vector dimension conflicts

### Constraints
- **HuggingFace Inference API** free tier has rate limits and occasional cold starts on less popular models
- **MultiQuery doubles token usage** — each question triggers multiple Groq calls (query generation + final answer)
- **No metadata populated** in `metadata_func` — source names fall back to file paths, limiting traceability of retrieved passages
- **Self-healing retriever** retries are redundant given the multi-query loop already broadens coverage